# Sanity check: imperfect classification in multiclass queues

This experiment verifies that the simulator correctly handles **latent (true) classes** and **observed (assigned) classes**.

Each arriving customer has:

- a **true class** $$C \in \{0,1\}$$
- an **assigned class** $$\hat C \in \{0,1\}$$

The assigned class is produced by a classifier with confusion matrix

$$
P(\hat C = j \mid C = i).
$$

Queueing decisions are made using the **assigned class**, while outcomes are evaluated using the **true class**.

In [1]:
import numpy as np
import pandas as pd

from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig
from gi_gi_n_gi_multiclass.dists.common import Exponential
from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS
from gi_gi_n_gi_multiclass.labelling import ConfusionMatrixClassifier
from gi_gi_n_gi_multiclass.metrics.class_metrics import (
    abandonment_probability_by_true_class,
    abandonment_probability_by_assigned_class,
    realized_confusion_table,
)
from gi_gi_n_gi_multiclass.outputs.tables import class_level_summary

## System configuration

We simulate a two-class overloaded system with

$$
N = 10
$$

servers and arrival rates

$$
\lambda_0 = 8, \qquad \lambda_1 = 12.
$$

Service and patience are exponential:

$$
\mu_0 = \mu_1 = 1, \qquad \theta_0 = \theta_1 = 0.5.
$$

Customers of assigned class 0 receive priority under a `PriorityFCFS` policy.

The key quantity of interest is

$$
P(\text{abandon} \mid \text{true chronic}),
$$

the probability that a truly chronic customer abandons the system.

In [2]:
N = 10
warmup_time = 500.0
run_time = 3000.0
seed = 123

# true class 0 = chronic
# true class 1 = transient
class0 = CustomerClass(
    name="chronic",
    interarrival=Exponential(rate=8.0),
    service=Exponential(rate=1.0),
    patience=Exponential(rate=0.5),
)

class1 = CustomerClass(
    name="transient",
    interarrival=Exponential(rate=12.0),
    service=Exponential(rate=1.0),
    patience=Exponential(rate=0.5),
)

classes = [class0, class1]

policy = PriorityFCFS(priority_order=[0, 1])   # prioritize assigned chronic

## Classifier scenarios

We test three levels of classifier accuracy.

### Perfect classifier

$$
P(\hat C = C) = 1
$$

$$
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

### 80% accurate classifier

$$
\begin{bmatrix}
0.8 & 0.2 \\
0.2 & 0.8
\end{bmatrix}
$$

### Uninformative classifier

$$
\begin{bmatrix}
0.5 & 0.5 \\
0.5 & 0.5
\end{bmatrix}
$$

The last case corresponds to a classifier with no information about the true class.

In [3]:
classifier_cases = {
    "perfect": np.array([
        [1.0, 0.0],
        [0.0, 1.0],
    ]),
    "accuracy_80": np.array([
        [0.8, 0.2],
        [0.2, 0.8],
    ]),
    "uninformative": np.array([
        [0.5, 0.5],
        [0.5, 0.5],
    ]),
}

## Realized confusion tables

The realized confusion tables match the target matrices closely.

For example, under the 80% classifier approximately

- 80% of true chronic customers are assigned chronic
- 80% of true transient customers are assigned transient

This confirms that the misclassification mechanism in the simulator is working correctly.

In [4]:
rows = []

for name, mat in classifier_cases.items():
    classifier = ConfusionMatrixClassifier(mat)

    cfg = MultiSimConfig(
        n_servers=N,
        classes=classes,
        classifier=classifier,
        policy=policy,
        seed=seed,
        warmup_time=warmup_time,
        run_time=run_time,
        collect_event_log=False,
    )

    res = run_sim_multi(cfg)

    rows.append({
        "classifier_case": name,
        "true_chronic_abandon_prob": abandonment_probability_by_true_class(res, 0),
        "true_transient_abandon_prob": abandonment_probability_by_true_class(res, 1),
        "assigned_chronic_abandon_prob": abandonment_probability_by_assigned_class(res, 0),
        "assigned_transient_abandon_prob": abandonment_probability_by_assigned_class(res, 1),
    })

    print("\n===", name, "===")
    print(realized_confusion_table(res))
    print(class_level_summary(res))

pd.DataFrame(rows)


=== perfect ===
assigned_class_id      0      1
true_class_id                  
0                  23944      0
1                      0  35818
     scope  arrivals  served  abandoned  lambda_hat  served_rate  \
0  OVERALL     59762   29837      29902   19.920667     9.945667   
1  class_0     23944   20927       3011    7.981333     6.975667   
2  class_1     35818    8910      26891   11.939333     2.970000   

   abandon_rate  Wq_mean_served  W_mean_served  L_timeavg  Lq_timeavg  
0      9.967333        0.891361       1.895731  29.752190   19.752191  
1      1.003667        0.251647       1.259921   9.014501    1.978283  
2      8.963667        2.393863       3.389065  20.725979   17.769306  

=== accuracy_80 ===
assigned_class_id      0      1
true_class_id                  
0                  19154   4790
1                   7233  28585
     scope  arrivals  served  abandoned  lambda_hat  served_rate  \
0  OVERALL     59762   30005      29731   19.920667    10.001667   
1  class_

,classifier_case,true_chronic_abandon_prob,true_transient_abandon_prob,assigned_chronic_abandon_prob,assigned_transient_abandon_prob
0,perfect,0.125752,0.750768,0.125752,0.750768
1,accuracy_80,0.269128,0.650148,0.139273,0.780704
2,uninformative,0.500209,0.497041,0.166277,0.831211


## Impact of classifier accuracy

The key study metric is

$$
P(\text{abandon} \mid \text{true chronic}).
$$

The results show a clear monotone relationship with classifier quality:

| classifier | chronic abandonment |
|------------|--------------------|
| perfect | 0.126 |
| 80% accurate | 0.269 |
| uninformative | 0.500 |

As the classifier becomes less accurate, the system becomes less able to protect truly chronic customers from abandonment.

## Interpretation

When classification is perfect, the priority policy correctly protects chronic customers.

As classifier accuracy declines, many chronic customers are misclassified into the lower-priority queue. These customers then compete with transient arrivals and experience higher abandonment risk.

At the extreme where the classifier is uninformative, priority is effectively assigned at random, and the protection of chronic customers disappears.

## System-level stability

Importantly, aggregate system metrics remain nearly unchanged across classifier settings:

- total arrival rate
- total service rate
- total abandonment rate
- average queue length

This shows that classification accuracy mainly affects **who receives service**, not overall system capacity.

## Conclusion

This experiment validates the new **latent-class simulation framework**.

The simulator now correctly models:

$$
\text{true class} \rightarrow \text{assigned class} \rightarrow \text{policy outcome}.
$$

Policy decisions are based on the assigned class, while performance can be evaluated using the true class.

This capability is essential for studying systems where priority decisions rely on **imperfect classification tools**, such as homelessness prioritization systems.